In [0]:
bronze_mobile_transactions_df = spark.table("workspace.banking_bronze.bronze_mobile_transactions")

# display(bronze_mobile_transactions_df)

bronze_mobile_transactions_df.printSchema() 

In [0]:
bronze_atm_transactions_df = spark.table("workspace.banking_bronze.bronze_atm_transactions")

# display(bronze_atm_transactions_df)

bronze_atm_transactions_df.printSchema()

In [0]:
bronze_branch_transactions_df  = spark.table("workspace.banking_bronze.bronze_branch_transactions")

# display(bronze_branch_transactions_df)
bronze_branch_transactions_df.printSchema()

In [0]:
bronze_mobile_transactions_df.printSchema() 
bronze_atm_transactions_df.printSchema()
bronze_branch_transactions_df.printSchema()


In [0]:
bronze_mobile_transactions_df = bronze_mobile_transactions_df.withColumnRenamed("source_file","_source_file")

In [0]:
bronze_mobile_transactions_df.printSchema() 
bronze_atm_transactions_df.printSchema()
bronze_branch_transactions_df.printSchema()


In [0]:
combined_mobile_atm_df = bronze_mobile_transactions_df.unionByName(bronze_atm_transactions_df)

In [0]:
display(combined_mobile_atm_df.count())

combined_mobile_atm_df.printSchema()

In [0]:
all_transactions_df = combined_mobile_atm_df.unionByName(bronze_branch_transactions_df)

In [0]:
display(all_transactions_df.count())

all_transactions_df.printSchema()

In [0]:
from pyspark.sql import functions as F
all_transactions_df.select(F.col("amount")).show(10)

In [0]:
all_transactions_df = all_transactions_df.withColumn("amount",
                                                     F.trim(F.col("amount")))
all_transactions_df.select("amount").show(10)
print("Record count: ",all_transactions_df.count())

In [0]:
all_transactions_df.select(
    F.col("amount"),
    F.length(F.col("amount")).alias("amount_length")
).show(10)

In [0]:
all_transactions_df = all_transactions_df.withColumn("amount",
    F.col("amount").cast("DECIMAL(18,2)")
)

all_transactions_df.printSchema()

In [0]:
all_transactions_df.select(F.col("transaction_timestamp")).show(10)

In [0]:
all_transactions_df = all_transactions_df.withColumn(
    "transaction_timestamp",
    F.to_timestamp(F.col("transaction_timestamp"))
)

all_transactions_df.printSchema()

In [0]:
# validating the null values in amount and transaction_timestamp
all_transactions_df.filter(
   (F.col("amount").isNull())
).count()

In [0]:
all_transactions_df.filter(
    F.col("transaction_timestamp").isNull()
).count()

In [0]:
all_transactions_df.filter(
    F.col("transaction_id").isNull()
).count()

all_transactions_df.filter(
    F.col("customer_id").isNull()
).count()

all_transactions_df.filter(
    F.col("account_id").isNull()
).count()


In [0]:
# Validate transaction types
all_transactions_df.select(
    F.col("transaction_type")
).distinct().show()

In [0]:
# Validating currency

all_transactions_df.select(
    F.col("currency")
).distinct().show()

In [0]:
# Inspect transaction events

mobile_events_df = spark.table("workspace.banking_bronze.bronze_mobile_events")

# display(mobile_events_df) 

atm_events_df = spark.table("workspace.banking_bronze.bronze_atm_events")
# display(atm_events_df) 

branch_events_df = spark.table("workspace.banking_bronze.bronze_branch_events")
# display(branch_events_df)

In [0]:
mobile_events_df.printSchema()

atm_events_df.printSchema()

branch_events_df.printSchema()

In [0]:
mobile_events_df = mobile_events_df.withColumnRenamed("source_file","_source_file")

atm_events_df = atm_events_df.withColumnRenamed("source_file","_source_file")


In [0]:
mobile_events_df = mobile_events_df.withColumn("event_sequence",
    F.col("event_sequence").cast("string")
)

mobile_events_df.printSchema()

In [0]:
all_events_df = mobile_events_df.unionByName(atm_events_df).unionByName(branch_events_df)

# display(all_events_df)

display(all_events_df.count())


In [0]:
all_events_df.select("event_timestamp").show(10)

In [0]:
all_events_df = all_events_df.withColumn("event_timestamp",
                F.to_timestamp(F.col("event_timestamp"))
                )
all_events_df.printSchema()

In [0]:
# Validate event timestamps

all_events_df.filter(
    F.col("event_timestamp").isNull()
).count()

In [0]:
# Inspect transaction statuses

all_events_df.select(
    F.col("status")
).distinct().show()

In [0]:
# Validate mandatory event fields

all_events_df.filter(
    F.col("transaction_id").isNull()
).count()

all_events_df.filter(
    F.col("event_id").isNull()
).count()

all_events_df.filter(
    F.col("event_sequence").isNull()
).count()

all_events_df.filter(
    F.col("status").isNull()
).count()

all_events_df.filter(
    F.col("event_timestamp").isNull()
).count()


In [0]:
# Validating duplicate values

all_events_df.groupBy("event_id")\
             .count()\
             .filter(F.col("count") > 1).show()

In [0]:
# Validate transaction references

orphan_events_df = all_events_df.join(
    all_transactions_df,
    "transaction_id",
    "left_anti"
)

orphan_events_df.count()

In [0]:
# Validate event sequence

all_events_df.select("event_sequence").distinct().show()

In [0]:
# Count events per transaction
all_events_df.groupBy("transaction_id")\
             .count()\
             .orderBy("transaction_id")\
             .show(50)

In [0]:
# Let's inspect the sequence numbers for each transaction.

all_events_df.groupBy("transaction_id")\
            .agg(
                F.collect_list("event_sequence").alias("event_sequence")
            )\
                .orderBy("transaction_id")\
                .show(50)

In [0]:
all_events_df.groupBy("transaction_id")\
             .agg(
                 F.min("event_sequence").alias("min_sequence"),
                 F.max("event_sequence").alias("max_sequence"),
                 F.count('event_sequence').alias("event_count")
             ).show()

In [0]:
# Find invalid sequences
event_sequence_validation_df = all_events_df.groupBy("transaction_id")\
                                            .agg(
                                                F.min("event_sequence").cast("int").alias("min_sequence"),
                                                F.max("event_sequence").cast("int").alias("max_sequence"),
                                                F.count("event_sequence").cast("int").alias("event_count")
                                            )


In [0]:
invalid_sequences_df = event_sequence_validation_df.filter(
    (F.col("max_sequence") - F.col("min_sequence") + 1)
    != F.col("event_count")
)

invalid_sequences_df.show()

In [0]:
# Check for duplicate event sequence numbers

all_events_df.groupBy("transaction_id","event_sequence")\
             .count()\
             .filter(F.col("count") > 1).show()

In [0]:
# Validate event sequence and timestamp order

all_events_df.orderBy("transaction_id",
             F.col("event_sequence").cast("int")
).select(
    "transaction_id",
    "event_sequence",
    "event_timestamp"
).show()

In [0]:
# Automatically detect incorrect timestamp order
from pyspark.sql.window import Window
from pyspark.sql import functions as F

event_window = Window.partitionBy("transaction_id")\
                     .orderBy(F.col("event_sequence").cast("int"))

timestamp_validation_df = all_events_df.withColumn("previous_timestamp",
                                                   F.lag("event_timestamp")
                                                   .over(event_window)
)

invalid_event_timestamps_df = timestamp_validation_df.filter(
    F.col("event_timestamp") <= F.col("previous_timestamp")

).select(
    "transaction_id",
    "event_sequence",
    "previous_timestamp",
    "event_timestamp"
).show()
                     

In [0]:
# Validate transaction-to-event relationships

transactions_without_events_df = all_transactions_df.join(
    all_events_df,
    "transaction_id",
    how="left_anti"
)

transactions_without_events_df.show()

In [0]:
all_events_df.groupBy("transaction_id")\
             .count()\
             .filter(F.col("count") > 1)\
             .orderBy("transaction_id")\
             .show()

In [0]:
# Inspect their status history

all_events_df.filter(
    F.col("transaction_id").isin(
        "TXN00003",
        "TXN00005",
        "TXN00011",
        "TXN00017",
        "TXN00026"
    )
).orderBy("transaction_id",
          F.col("event_sequence").cast("int")
          ).select(
              "transaction_id",
              "event_sequence",
              "status",
              "event_timestamp"
          ).show()

In [0]:
latest_event_window = Window.partitionBy("transaction_id")\
                            .orderBy(F.col("event_sequence").cast("int").desc())

ranked_events_df = all_events_df.withColumn("row_number",
                                            F.row_number().over(latest_event_window)
                                            )

ranked_events_df.filter(
    F.col("transaction_id") == "TXN00003"
).select(
    "transaction_id",
    "event_sequence",
    "status",
    "row_number"
).show()    

In [0]:
latest_event_df = ranked_events_df.filter(
    F.col("row_number") == 1
)

latest_event_df.select(
    "transaction_id",
    "event_sequence",
    "status",
    "event_timestamp"
).show()

In [0]:
latest_event_df.count()

In [0]:
silver_transactions_df = all_transactions_df.join(
    latest_event_df.select(
        "transaction_id",
        "status",
        "event_timestamp"
    ),
    "transaction_id",
    "inner"
)

silver_transactions_df.count()

In [0]:
silver_transactions_df.show(10)

In [0]:
silver_transactions_df.groupBy("status")\
                     .count()\
                     .orderBy("status")\
                     .show()

In [0]:
# Validate the status values

silver_transactions_df.select("status")\
                      .distinct().show()

In [0]:
# Check duplicate transactions

silver_transactions_df.groupBy("transaction_id")\
                      .count()\
                      .filter(F.col("count") > 1).show()

In [0]:
# Let's check whether any transaction has an amount less than or equal to zero.

silver_transactions_df.filter(
    F.col("amount") <= 0
).select(
    "transaction_id",
    "amount",
    "status"
).show()

In [0]:
# Validate customer and account relationships

invalid_customers_df = silver_transactions_df.join(
    spark.table("workspace.banking_bronze.bronze_customers"),
    "customer_id",
    "left_anti"

)

invalid_customers_df.select(
    "transaction_id",
    "customer_id"
).show()

In [0]:
invalid_accounts_df = silver_transactions_df.join(
    spark.table("workspace.banking_bronze.bronze_accounts"),
    "account_id",
    "left_anti"
)
invalid_accounts_df.select(
    "transaction_id",
    "account_id"
).show()

In [0]:
# Validate transaction types
silver_transactions_df.select("transaction_type") \
    .distinct() \
    .show()

In [0]:
# Validate currency
silver_transactions_df.select("currency")\
                      .distinct()\
                      .show()

In [0]:
# Validate transaction timestamps against event timestamps

silver_transactions_df.filter(
    F.col("event_timestamp") < F.col("transaction_timestamp")
).select(
    "transaction_id",
    "transaction_timestamp",
    "event_timestamp",
    "status"
).show()

In [0]:
# Check for NULL values in the final Silver dataset

silver_transactions_df.select([
    F.count(
        F.when(F.col(c).isNull(),c)
    ).alias(c)
    for c in [
         "transaction_id",
        "customer_id",
        "account_id",
        "transaction_type",
        "amount",
        "currency",
        "transaction_timestamp",
        "status",
        "event_timestamp"
    ]
]).show()

In [0]:
silver_transactions_df.write\
                      .format("delta")\
                      .mode("overwrite")\
                     .saveAsTable("workspace.banking_silver.silver_transactions")

In [0]:
display(spark.table("workspace.banking_silver.silver_transactions"))
display(spark.table("workspace.banking_silver.silver_transactions").count())


In [0]:
all_events_df.printSchema()

In [0]:
all_events_df = all_events_df.withColumn("event_sequence",
    F.col("event_sequence").cast("int")
)



In [0]:
all_events_df.printSchema()

In [0]:
# Validate event sequence after conversion

all_events_df.filter(
    F.col("event_sequence").isNull()
).count()

In [0]:
# Validate event status values

all_events_df.select("status")\
             .distinct().show()

In [0]:
# Validate duplicate event IDs

all_events_df.groupBy("event_id")\
             .count()\
             .filter(F.col("count") > 1).show()

In [0]:
# Validate NULL values in Silver Events

all_events_df.select([
    F.count(
        F.when(F.col(c).isNull(),c)
    ).alias(c)
    for c in [
        "transaction_id",
        "event_id",
        "event_sequence",
        "status",
        "event_timestamp"
    ]
]).show()

In [0]:
# Validate event-to-transaction relationships again

invalid_events_df = all_events_df.join(
    silver_transactions_df.select(
        "transaction_id"
    ),
    "transaction_id",
    "left_anti"
)

invalid_events_df.show()

In [0]:
all_events_df.count()

In [0]:
all_events_df.write\
             .format("delta")\
             .mode("overwrite")\
             .saveAsTable("workspace.banking_silver.silver_events")

In [0]:
spark.table("workspace.banking_silver.silver_events").count()

In [0]:
bronze_customers_df = spark.table("workspace.banking_bronze.bronze_customers")

bronze_accounts_df = spark.table("workspace.banking_bronze.bronze_accounts")

bronze_customers_df.printSchema()
bronze_accounts_df.printSchema()

In [0]:
print("Customers record count: ",bronze_customers_df.count())
print("Accounts record count: ",bronze_accounts_df.count())

In [0]:
# Check duplicate IDs
from pyspark.sql import functions as F
bronze_customers_df.groupBy("customer_id")\
                   .count()\
                    .filter(F.col("count") > 1).show()


bronze_accounts_df.groupBy("account_id")\
                   .count()\
                    .filter(F.col("count") > 1).show()

In [0]:
# Check NULL values in key columns
bronze_customers_df.filter(
    F.col("customer_id").isNull()
).count()

bronze_accounts_df.filter(
    F.col("account_id").isNull()
).count()

bronze_accounts_df.filter(
    F.col("customer_id").isNull()
).count()

In [0]:
# Validate account-to-customer relationships

invalid_accounts_df = bronze_accounts_df.join(
    bronze_customers_df,
    "customer_id",
    "left_anti"
)

invalid_accounts_df.show()

In [0]:
# Check customer segment values
bronze_customers_df.select(
    F.col("customer_segment")
).distinct().show()

In [0]:
# Check NULL values in customer_segment and account_type also check does account type is distinct

bronze_customers_df.filter(
    F.col("customer_segment").isNull()
).count()

bronze_accounts_df.filter(
    F.col("account_type").isNull()
).count()

bronze_accounts_df.select(
    F.col("account_type")
).distinct().show()

In [0]:
# Check whether every customer has at least one account

customers_without_account = bronze_customers_df.join(
    bronze_accounts_df,
    "customer_id",
    "left_anti"
)

customers_without_account.show()

In [0]:
# Check account distribution per customer
bronze_accounts_df.groupBy("customer_id")\
                  .count()\
                  .orderBy("customer_id")\
                  .show()

In [0]:
bronze_accounts_df = bronze_accounts_df.withColumnRenamed("_ingestion_time","_ingestion_timestamp")

In [0]:
bronze_accounts_df.printSchema()

In [0]:
# Create the Silver Customers DataFrame
silver_customers_df = bronze_customers_df.select(
    "customer_id",
    "customer_segment",
    "source_file",
    "_ingestion_timestamp"
)

In [0]:
silver_customers_df.show()
silver_customers_df.count()

In [0]:
# Create the Silver Accounts DataFrame

silver_accounts_df = bronze_accounts_df.select(
    "account_id",
    "customer_id",
    "account_type",
    "source_file",
    "_ingestion_timestamp"
)

silver_accounts_df.show()
silver_accounts_df.count()

In [0]:
# Final NULL validation in Silver DataFrames
silver_customers_df.filter(
    F.col("customer_id").isNull() |
    F.col("customer_segment").isNull()
).count()

In [0]:
silver_accounts_df.filter(
    F.col("account_id").isNull() |
    F.col("customer_id").isNull() |
    F.col("account_type").isNull()
).count()

In [0]:
# Save Silver Customers as a Delta table
silver_customers_df.write\
                   .format("delta")\
                   .mode("overwrite")\
                   .saveAsTable("workspace.banking_silver.silver_customers")

In [0]:
spark.table("workspace.banking_silver.silver_customers").count()

In [0]:
# Save Silver Accounts as a Delta table
silver_accounts_df.write\
                  .format("delta")\
                  .mode("overwrite")\
                  .saveAsTable("workspace.banking_silver.silver_accounts")

In [0]:
spark.table("workspace.banking_silver.silver_accounts").count()
spark.table("workspace.banking_silver.silver_accounts").display()


In [0]:
# Join Transactions with Accounts
silver_transactions_df = spark.table("workspace.banking_silver.silver_transactions")

transaction_account_df = silver_transactions_df.join(
    silver_accounts_df.select(
    "account_id",
    F.col("customer_id").alias("account_customer_id")
    ),
    "account_id",
    "inner"
)

In [0]:
invalid_ownership_df = transaction_account_df.filter(
    F.col("customer_id") != F.col("account_customer_id")
)

invalid_ownership_df.select(
    "transaction_id",
    "customer_id",
    "account_id",
    "account_customer_id"
).show()

In [0]:
# Verify transaction count after the join
transaction_account_df.count()

In [0]:

silver_transactions_df = spark.table("workspace.banking_silver.silver_transactions")

silver_transactions_df.columns



In [0]:
# Identify invalid transactions
from pyspark.sql import functions as F

invalid_transactions_df = silver_transactions_df.filter(
    F.col("transaction_id").isNull() |
    F.col("customer_id").isNull() |
    F.col("account_id").isNull() |
    F.col("amount").isNull() |
    (F.col("amount") <= 0) |
    (~F.col("transaction_type").isin(
        "DEPOSIT", "WITHDRAWAL", "TRANSFER", "PAYMENT"
    )) |
    (F.col("currency") != "INR") |
    (~F.col("status").isin(
        "SUCCESS", "FAILED", "PENDING", "REVERSED"
    )) |
    (F.col("event_timestamp") < F.col("transaction_timestamp"))
)

print("Invalid transaction count:", invalid_transactions_df.count())

In [0]:
valid_transactions_df = silver_transactions_df.filter(
    ~F.col("transaction_id").isNull() &
    ~F.col("customer_id").isNull() &
    ~F.col("account_id").isNull() &
    ~F.col("amount").isNull() &
    (F.col("amount") > 0) |
    (F.col("transaction_type").isin(
        "DEPOSIT", "WITHDRAWAL", "TRANSFER", "PAYMENT"
    )) |
    (F.col("currency") == "INR") |
    (F.col("status").isin(
        "SUCCESS", "FAILED", "PENDING", "REVERSED"
    )) |
    (F.col("event_timestamp") >= F.col("transaction_timestamp"))
)

quarantine_transactions_df = invalid_transactions_df


In [0]:
print("Valid transacations: ",valid_transactions_df.count())
print("Invalid transactions: ",quarantine_transactions_df.count())

In [0]:
quarantine_transactions_df = invalid_transactions_df.withColumn(
    "rejection_reason",
    F.concat_ws(
        " | ",
        F.when(F.col("transaction_id").isNull(),"Missing_transaction_id"),
        F.when(F.col("customer_id").isNull(),"Missing_customer_id"),
        F.when(F.col("account_id").isNull(),"Missing_account_id"),
        F.when(F.col("amount").isNull(),"Missing_amount"),
        F.when(F.col("amount") <= 0,"Invalid_amount"),
        F.when(
            ~F.col("transaction_type").isin(
                "DEPOSIT","WITHDRAWAL","TRANSFER","PAYMENT"
            ),"Invalid_transaction_type"    
        ),
        F.when(F.col("currency") != "INR","Invalid_currency"),
        F.when(
            ~F.col("status").isin(
                "SUCCESS","FAILED","PENDING","REVERSED"
            ),"Invalid_status"
        ),
        F.when(F.col("event_timestamp") < F.col("transaction_timestamp"),"Invalid_event_timestamp")
    )
)

In [0]:
quarantine_transactions_df.printSchema()

In [0]:
quarantine_transactions_df.select(
    "transaction_id",
    "amount",
    "currency",
    "rejection_reason"
).show(10)

In [0]:
quarantine_transactions_df.write\
                          .format("delta")\
                          .mode("overwrite")\
                          .saveAsTable("workspace.banking_quarantine.quarantine_transactions")

In [0]:
silver_events_df = spark.table("workspace.banking_silver.silver_events")

silver_events_df.count()

In [0]:
# Identify invalid events
invalid_events_df = silver_events_df.filter(
    F.col("transaction_id").isNull() |
    F.col("event_id").isNull() |
    F.col("event_sequence").isNull() |
    (F.col("event_sequence") <= 0) |
    (~F.col("status").isin(
        "SUCCESS","FAILED","PENDING","REVERSED"
    )) |
    F.col("event_timestamp").isNull()
)

print("Invalid event count: ",invalid_events_df.count())

In [0]:
# validating for transaction and event relationship
orphan_events_df = silver_events_df.join(
    silver_transactions_df.select("transaction_id"),
    "transaction_id",
    "left_anti"
)

print("Orphan event count: ",orphan_events_df.count())

In [0]:
# Create the valid and invalid event DataFrames

valid_events_df = silver_events_df.filter(
    ~F.col("transaction_id").isNull() &
    ~F.col("event_id").isNull() &
    ~F.col("event_sequence").isNull() &
    (F.col("event_sequence") > 0) &
    (F.col("status").isin(
        "SUCCESS","FAILED","PENDING","REVERSED"
    )) |
    ~F.col("event_timestamp").isNull()
).join(
    silver_transactions_df.select("transaction_id"),
    "transaction_id",
    "inner"
)
quarantine_events_df = invalid_events_df.unionByName(orphan_events_df)

In [0]:
print("valid events: ",valid_events_df.count())
print("quarantine events: ",quarantine_events_df.count())


In [0]:
# Add rejection reasons to quarantined events

quarantine_events_df = quarantine_events_df.withColumn(
    "Rejection_reason",
    F.concat_ws(" | ",
                F.when(F.col("transaction_id").isNull(),"Missing_transaction_id"),
                F.when(F.col("event_id").isNull(),"Missing_event_id"),
                F.when(F.col("event_sequence").isNull(),"Missing_event_sequence"),
                F.when(F.col("event_sequence") <= 0,"Invalid_event_sequence"),
                F.when(~F.col("status").isin("SUCCESS","FAILED","PENDING","REVERSED"),"Invalid_status"),
                F.when(F.col("event_timestamp").isNull(),"Missing_event_timestamp")
                )
)

In [0]:
quarantine_events_df.printSchema()
print(quarantine_events_df.count())

In [0]:
quarantine_events_df = quarantine_events_df.withColumnRenamed("Rejection_reason","rejection_reason")

quarantine_events_df.printSchema()
print(quarantine_events_df.count())

In [0]:
quarantine_events_df.write\
                    .format("delta")\
                    .mode("overwrite")\
                    .saveAsTable("workspace.banking_quarantine.quarantine_events")

In [0]:
display(spark.table("workspace.banking_quarantine.quarantine_events").count())  

In [0]:
# Customers quarantine validation

silver_customers_df = spark.table("workspace.banking_silver.silver_customers")

print("silver customers count: ",silver_customers_df.count())

In [0]:
# Check missing customer IDs
silver_customers_df.filter(
    F.col("customer_id").isNull()
).count()

In [0]:
# Check duplicate customer IDs
silver_customers_df.groupBy("customer_id")\
                   .count()\
                   .filter(F.col("count") > 1).count()

In [0]:
# Validate customer segments
silver_customers_df.filter(
    F.col("customer_segment").isNull() |
    (~F.col("customer_segment").isin("RETAIL","PREMIUM","SME"))
).count()

In [0]:
valid_customers_df = silver_customers_df.filter(
    ~F.col("customer_id").isNull() &
    F.col("customer_segment").isin("RETAIL","PREMIUM","SME")
)

quarantine_customers_df = silver_customers_df.filter(
    F.col("customer_id").isNull() |
    (~F.col("customer_segment").isin("RETAIL","PREMIUM","SME"))
)

print("valid customers count: ",valid_customers_df.count())
print("quarantine customers count: ",quarantine_customers_df.count())


In [0]:
# Add rejection reasons

quarantine_customers_df = quarantine_customers_df.withColumn(
    "rejection_reason",
    F.concat_ws(" | ",
                F.when(F.col("customer_id").isNull(),"Missing_customer_id"),
                F.when(F.col("customer_segment").isNull(),"Missing_customer_segment"),
                F.when(~F.col("customer_segment").isin("RETAIL","PREMIUM","SME"),"Invalid_customer_segment")
                )
)

In [0]:
quarantine_customers_df.printSchema()
print(quarantine_customers_df.count())

In [0]:
quarantine_customers_df.write\
                       .format("delta")\
                       .mode("overwrite")\
                       .saveAsTable("workspace.banking_quarantine.quarantine_customers")

In [0]:
display(spark.table("workspace.banking_quarantine.quarantine_customers").count())

In [0]:
silver_accounts_df = spark.table("workspace.banking_silver.silver_accounts")

print("silver accounts count: ",silver_accounts_df.count())



In [0]:
# Accounts quarantine validation
# Check missing Account IDs

silver_accounts_df.filter(
    F.col("account_id").isNull()
).count()

In [0]:
# Check duplicate Account IDs
silver_accounts_df.groupBy("account_id")\
                  .count()\
                  .filter(F.col("count") > 1).count()

In [0]:
# Check missing Customer IDs
silver_accounts_df.filter(
    F.col("customer_id").isNull()
).count()

In [0]:
# Validate Account Types
silver_accounts_df.filter(
    F.col("account_type").isNull() |
    (~F.col("account_type").isin("SAVINGS", "CURRENT"))
).count()

In [0]:
# Validate customer references
orphan_accounts_df = silver_accounts_df.join(
    silver_customers_df.select("customer_id"),
    "customer_id",
    "left_anti"
)

print("orphan accounts count: ",orphan_accounts_df.count())

In [0]:
valid_accounts_df = silver_accounts_df.filter(
    ~F.col("account_id").isNull() &
    ~F.col("customer_id").isNull() &
    F.col("account_type").isin("SAVINGS","CURRENT")
).join(
    silver_customers_df.select("customer_id"),
    "customer_id",
    "inner"
)

invalid_accounts_df = silver_accounts_df.filter(
    F.col("account_id").isNull() |
    F.col("customer_id").isNull() |
    (~F.col("account_type").isin("SAVINGS","CURRENT")) |
    F.col("account_type").isNull()
)

quarantine_accounts_df = invalid_accounts_df.unionByName(orphan_accounts_df).dropDuplicates(["account_id"])

print("Valid accounts: ",valid_accounts_df.count())
print("Quarantine accounts: ",quarantine_accounts_df.count())

In [0]:
# Add rejection reasons

quarantine_accounts_df = quarantine_accounts_df.withColumn(
    "rejection_reason",
    F.concat_ws(" | ",
                F.when(F.col("account_id").isNull(),"Missing_account_id"),
                F.when(F.col("customer_id").isNull(),"Missing_customer_id"),
                F.when(F.col("account_type").isNull(),"Missing_account_type"),
                F.when(~F.col("account_type").isin("SAVINGS","CURRENT"),"Invalid_account_type"),
                F.when(
                    F.col("customer_id").isNotNull() &
                    (~F.col("customer_id").isin(
                      [row.customer_id for row in silver_customers_df.select("customer_id").collect()])),"Orphan_customer_reference")
                )
)

In [0]:
quarantine_accounts_df.printSchema()
print(quarantine_accounts_df.count())

In [0]:
# Save the Accounts quarantine table
quarantine_accounts_df.write\
                      .format("delta")\
                      .mode("overwrite")\
                      .saveAsTable("workspace.banking_quarantine.quarantine_accounts")

In [0]:
display(spark.table("workspace.banking_quarantine.quarantine_accounts").count())
